# Phase 2: DistilBERT Intent Classifier Training
## SRH Chatbot — MSc AI/ML Dissertation

---

### What this notebook does

Fine-tunes `distilbert-base-uncased` for **multi-label intent classification**
on annotated SRH queries. Each query can belong to one or more of three
categories simultaneously:

| Label | Meaning |
|---|---|
| `educational` | General SRH information request |
| `diagnostic` | Symptom or personal health concern |
| `crisis` | Abuse, assault, danger, or emotional distress |

The trained model replaces the keyword-based `_detect_type()` function in
`src/safety/escalation.py`. The **public interface** (`should_escalate`,
`get_escalation_response`) stays completely unchanged.

---

### How to use this notebook

1. Open in Google Colab
2. **Runtime → Change runtime type → T4 GPU** (required for reasonable speed)
3. In Cell 4, replace `YOUR_USERNAME` with your actual GitHub username
4. Run all cells top to bottom
5. The trained model is saved to `models/intent_classifier/` and pushed to GitHub

### Swapping in real data

When annotation is complete, run `merge_annotations.py` to produce
`data/raw_queries/merged_annotations.csv`, then re-run from Cell 6 downward.
No code changes are needed — Cell 6 detects the file automatically.

### Expected training time (T4 GPU)

~5–10 minutes for 5 epochs on ~200 queries.

### Output

`models/intent_classifier/` containing:
- `pytorch_model.bin` — trained weights
- `config.json`, `tokenizer_config.json`, `vocab.txt` — model config
- `classifier_config.json` — labels, crisis threshold, evaluation results


## 1. GPU Check

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected.")
    print("Go to: Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU")
    print("Continuing on CPU (slow but works for testing).")

print(f"\nUsing device: {device}")


## 2. Install Dependencies

In [ ]:
# -U (not a pinned old version) so transformers stays compatible with
# whatever accelerate/tokenizers Colab already has preinstalled — pinning
# transformers alone to an old version broke Trainer with:
# "ImportError: cannot import name 'EncoderDecoderCache' from 'transformers'"
!pip install -q -U transformers accelerate datasets scikit-learn seaborn matplotlib
print("All packages ready.")


## 3. Clone Repository

> **Before running:** replace `YOUR_USERNAME` with your actual GitHub username.


In [ ]:
import os

if not os.path.exists("/content/srh-chatbot"):
    # Replace YOUR_USERNAME with your actual GitHub username
    !git clone https://github.com/abiolalawal14/Abiyamo.git /content/srh-chatbot
else:
    print("Repository already cloned — pulling latest.")
    !git -C /content/srh-chatbot pull

%cd /content/srh-chatbot

import sys
sys.path.insert(0, "/content/srh-chatbot")
print("Repository ready. Python path updated.")


## 4. Configuration

In [ ]:
import os

# All configurable values live here — every other cell references these
# variables so a single change here propagates everywhere.

DATA_PATH  = "data/raw_queries/training_dataset.csv"
MODEL_NAME = "distilbert-base-uncased"
OUTPUT_DIR = "models/intent_classifier"
LABELS     = ["educational", "diagnostic", "crisis"]

MAX_LENGTH    = 128
BATCH_SIZE    = 16
NUM_EPOCHS    = 5
LEARNING_RATE = 2e-5
TEST_SIZE     = 0.15
VAL_SIZE      = 0.15
RANDOM_SEED   = 42

# Crisis recall must reach this threshold before the model is considered
# safe to deploy. Missing a crisis query routes a vulnerable user to an
# educational response instead of a helpline — false negatives here have
# real safety consequences. 0.80 is conservative but reachable for a
# dataset of this size.
CRISIS_RECALL_THRESHOLD = 0.80

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  Labels : {LABELS}")
print(f"  Model  : {MODEL_NAME}")
print(f"  Output : {OUTPUT_DIR}")


## 5. Load and Prepare Data

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH)

# training_dataset.csv already has binary columns —
# no parsing needed unlike merged_annotations.csv
# which required extracting labels from annotator columns
print(f"Loaded {len(df)} training examples from {DATA_PATH}")
print(f"Columns: {list(df.columns)}")
print(f"\nLabel distribution:")
for label in LABELS:
    count = df[label].sum()
    pct = count / len(df) * 100
    print(f"  {label}: {int(count)} ({pct:.1f}%)")
print(f"\nSource breakdown:")
print(df['source'].value_counts().to_string())


## 6. Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify on the crisis column specifically — it is the smallest and
# most important class. Without stratification, a random split could
# leave the test set with very few crisis examples, making evaluation
# of the most safety-critical label unreliable.
train_df, temp_df = train_test_split(
    df,
    test_size=TEST_SIZE + VAL_SIZE,
    stratify=df["crisis"],
    random_state=RANDOM_SEED,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE),
    stratify=temp_df["crisis"],
    random_state=RANDOM_SEED,
)

# Reset indices so positional indexing works correctly in the Dataset class
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("Split sizes:")
print(f"  Train      : {len(train_df)} ({100*len(train_df)/len(df):.0f}%)")
print(f"  Validation : {len(val_df)} ({100*len(val_df)/len(df):.0f}%)")
print(f"  Test       : {len(test_df)} ({100*len(test_df)/len(df):.0f}%)")
print()
print("Crisis distribution across splits:")
for name, split in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    c = int(split["crisis"].sum())
    print(f"  {name:<5}: {c} crisis ({100*c/len(split):.1f}%)")


## 7. Tokenisation

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")


def tokenize(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors=None,  # return plain lists; Dataset converts to tensors
    )


train_enc = tokenize(train_df["text"].tolist())
val_enc   = tokenize(val_df["text"].tolist())
test_enc  = tokenize(test_df["text"].tolist())

train_labels = train_df[LABELS].values
val_labels   = val_df[LABELS].values
test_labels  = test_df[LABELS].values


class SRHIntentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        # float32 is required by BCEWithLogitsLoss, which is the correct
        # loss for multi-label classification — it applies sigmoid
        # independently per label rather than softmax across all labels.
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item


train_dataset = SRHIntentDataset(train_enc, train_labels)
val_dataset   = SRHIntentDataset(val_enc,   val_labels)
test_dataset  = SRHIntentDataset(test_enc,  test_labels)

print(f"Datasets created:")
print(f"  train={len(train_dataset)}, val={len(val_dataset)}, test={len(test_dataset)}")


## 8. Model Definition

In [ ]:
from transformers import DistilBertForSequenceClassification

# problem_type="multi_label_classification" tells HuggingFace to:
#   1. Use BCEWithLogitsLoss (sigmoid per label, not softmax across labels)
#   2. Expect float32 labels (0.0 / 1.0), not integer class indices
#
# This is correct for our task: a query can be BOTH educational AND
# diagnostic simultaneously. Softmax would force exactly one label,
# which is the wrong assumption.
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type="multi_label_classification",
)
model = model.to(device)

print(f"Model loaded: {MODEL_NAME}")
print(f"  Labels     : {LABELS}")
print(f"  Output head: {len(LABELS)} logits (sigmoid applied per label)")
print(f"  Device     : {device}")


## 9. Training

In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from transformers import TrainingArguments, Trainer


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Sigmoid converts raw logits to per-label probabilities [0, 1].
    # Threshold at 0.5 — Cell 12 tunes this for crisis specifically.
    probs  = 1 / (1 + np.exp(-logits))
    preds  = (probs >= 0.5).astype(int)
    labels = labels.astype(int)

    results = {}
    for i, label_name in enumerate(LABELS):
        results[f"{label_name}_precision"] = precision_score(labels[:, i], preds[:, i], zero_division=0)
        results[f"{label_name}_recall"]    = recall_score(labels[:, i], preds[:, i], zero_division=0)
        results[f"{label_name}_f1"]        = f1_score(labels[:, i], preds[:, i], zero_division=0)

    results["micro_f1"] = f1_score(labels, preds, average="micro", zero_division=0)

    crisis_idx    = LABELS.index("crisis")
    crisis_recall = recall_score(labels[:, crisis_idx], preds[:, crisis_idx], zero_division=0)
    if crisis_recall < CRISIS_RECALL_THRESHOLD:
        print(f"  WARNING: crisis recall {crisis_recall:.3f} below threshold {CRISIS_RECALL_THRESHOLD:.2f}")

    return results


training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=10,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),  # FP16 only on GPU; causes errors on CPU
    report_to="none",                # suppress wandb / tensorboard prompts
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("\nTraining complete.")


## 10. Evaluation on Test Set

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score,
)

test_output = trainer.predict(test_dataset)
logits = test_output.predictions
labels = test_output.label_ids.astype(int)

probs = 1 / (1 + np.exp(-logits))
preds = (probs >= 0.5).astype(int)

# ── 1. Per-label metrics table ───────────────────────────────────────────────
print("=" * 65)
print("  TEST SET EVALUATION")
print("=" * 65)
print(f"  {'Label':<15} {'Precision':>10} {'Recall':>8} {'F1':>8} {'Support':>9}")
print("  " + "-" * 53)

crisis_recall_val = None
for i, label in enumerate(LABELS):
    p = precision_score(labels[:, i], preds[:, i], zero_division=0)
    r = recall_score(labels[:, i], preds[:, i], zero_division=0)
    f = f1_score(labels[:, i], preds[:, i], zero_division=0)
    s = int(labels[:, i].sum())
    print(f"  {label:<15} {p:>10.3f} {r:>8.3f} {f:>8.3f} {s:>9}")
    if label == "crisis":
        crisis_recall_val = r

micro_f1 = f1_score(labels, preds, average="micro", zero_division=0)
print(f"\n  Micro-averaged F1: {micro_f1:.3f}")

# ── 2. Crisis recall prominence ──────────────────────────────────────────────
print()
print("=" * 65)
if crisis_recall_val >= CRISIS_RECALL_THRESHOLD:
    print(f"  CRISIS RECALL: {crisis_recall_val:.3f}  --  PASS")
else:
    print(f"  CRISIS RECALL: {crisis_recall_val:.3f}  --  BELOW THRESHOLD")
    print(f"  Threshold required: {CRISIS_RECALL_THRESHOLD:.2f}")
    print()
    print("  RECOMMENDATION: Lower the crisis classification threshold")
    print("  from 0.5 to 0.3 or 0.4. See Cell 11 (Threshold Tuning).")
print("=" * 65)

# ── 3. Confusion matrices (one 2x2 per label) ────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Confusion Matrices — Test Set", fontsize=14)

for i, label in enumerate(LABELS):
    cm = confusion_matrix(labels[:, i], preds[:, i])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"],
        ax=axes[i],
    )
    axes[i].set_title(label.capitalize())

plt.tight_layout()
plt.show()

# ── 4. sklearn classification report ─────────────────────────────────────────
print("\nFull classification report:")
for i, label in enumerate(LABELS):
    print(f"\n  [{label.upper()}]")
    print(classification_report(
        labels[:, i], preds[:, i],
        target_names=[f"not_{label}", label],
        zero_division=0,
    ))


## 11. Threshold Tuning for Crisis

In [ ]:
crisis_idx   = LABELS.index("crisis")
crisis_probs = probs[:, crisis_idx]
crisis_true  = labels[:, crisis_idx]

thresholds = np.arange(0.20, 0.71, 0.05)

print("Crisis label — threshold sensitivity:")
print(f"  {'Threshold':>10} {'Precision':>11} {'Recall':>8} {'F1':>8}")
print("  " + "-" * 42)

best_f1 = -1.0
recommended_threshold = None

for thresh in thresholds:
    preds_t = (crisis_probs >= thresh).astype(int)
    p = precision_score(crisis_true, preds_t, zero_division=0)
    r = recall_score(crisis_true, preds_t, zero_division=0)
    f = f1_score(crisis_true, preds_t, zero_division=0)

    # Best = highest F1 among thresholds that also achieve target recall.
    # Prioritising recall over F1 here because false negatives (missed
    # crisis queries) are more harmful than false positives.
    meets = "  <-- recall >= 0.80" if r >= CRISIS_RECALL_THRESHOLD else ""
    print(f"  {thresh:>10.2f} {p:>11.3f} {r:>8.3f} {f:>8.3f}{meets}")

    if r >= CRISIS_RECALL_THRESHOLD and f > best_f1:
        best_f1 = f
        recommended_threshold = float(thresh)

print()
if recommended_threshold is None:
    recommended_threshold = float(thresholds[0])
    print(f"NOTE: No threshold achieves recall >= {CRISIS_RECALL_THRESHOLD:.2f}.")
    print("Consider collecting more crisis examples and retraining.")
    print(f"Using lowest threshold ({recommended_threshold:.2f}) as a fallback.")
else:
    print(f"Recommended crisis threshold: {recommended_threshold:.2f}")
    print(f"Achieves recall >= {CRISIS_RECALL_THRESHOLD:.2f} while maximising F1.")

print()
print("This threshold replaces 0.5 in escalation._detect_type() when")
print("the DistilBERT classifier is integrated into the chatbot pipeline.")

# Store for Cell 12 (Save)
OPTIMAL_CRISIS_THRESHOLD = recommended_threshold


## 12. Save Model and Tokenizer

> **Before running:** replace the `git config` values with your actual
> GitHub email and name.


In [ ]:
import json
from datetime import date

# Save model weights and tokenizer files
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model and tokenizer saved to {OUTPUT_DIR}/")

# classifier_config.json is what escalation.py reads when loading the
# DistilBERT classifier to replace _detect_type(). It carries the
# per-deployment decisions (threshold, labels) alongside the evaluation
# results so the config is self-documenting.
classifier_config = {
    "model_name": MODEL_NAME,
    "labels": LABELS,
    "crisis_threshold": OPTIMAL_CRISIS_THRESHOLD,
    "max_length": MAX_LENGTH,
    "trained_on": str(date.today()),
    "dataset_size": int(len(df)),
    "crisis_recall": float(crisis_recall_val),
}

config_path = os.path.join(OUTPUT_DIR, "classifier_config.json")
with open(config_path, "w") as f:
    json.dump(classifier_config, f, indent=2)

print(f"Classifier config saved to {config_path}:")
print(json.dumps(classifier_config, indent=2))

# Push saved model to GitHub so weights are versioned alongside the code.
# Replace these values with your actual GitHub credentials.
!git config user.email "your_email@example.com"
!git config user.name "Your Name"
!git add models/intent_classifier/
!git commit -m "Add trained DistilBERT intent classifier"
!git push


## 13. Inference Test

In [ ]:
from transformers import AutoTokenizer, DistilBertForSequenceClassification
import torch, json, os

# Load from the saved directory — verifies the artifacts are self-contained
# and work independently of the training session objects above.
saved_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
saved_model     = DistilBertForSequenceClassification.from_pretrained(OUTPUT_DIR)
saved_model     = saved_model.to(device)
saved_model.eval()

with open(os.path.join(OUTPUT_DIR, "classifier_config.json")) as f:
    saved_config = json.load(f)

crisis_threshold = saved_config["crisis_threshold"]
print(f"Model loaded from {OUTPUT_DIR}")
print(f"Crisis threshold: {crisis_threshold}\n")


def predict(text, crisis_thresh=crisis_threshold):
    """Run inference on a single text string."""
    inputs = saved_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding=True,
    ).to(device)

    with torch.no_grad():
        logits = saved_model(**inputs).logits[0]

    probs = torch.sigmoid(logits).cpu().numpy()

    # Apply per-label thresholds: crisis uses a lower threshold than
    # the other labels to favour recall over precision for the
    # safety-critical category.
    predicted = []
    for i, label in enumerate(LABELS):
        thresh = crisis_thresh if label == "crisis" else 0.5
        if probs[i] >= thresh:
            predicted.append(label)

    return predicted if predicted else ["educational"]  # safe default


test_queries = [
    ("How do I use a condom correctly?",                      "educational"),
    ("What are the types of contraception?",                  "educational"),
    ("I have had unusual discharge for two weeks",            "diagnostic"),
    ("My period has been irregular for three months",         "diagnostic"),
    ("I was forced to have sex and I don't know what to do", "crisis"),
    ("I am pregnant and I am scared, I need help urgently",  "crisis"),
]

print(f"{'Query':<54} {'Predicted':<25} {'Expected':<14} Result")
print("-" * 108)

correct = 0
for text, expected in test_queries:
    predicted = predict(text)
    match     = expected in predicted
    correct  += int(match)
    result    = "PASS" if match else "FAIL"
    print(f"{text[:52]:<54} {str(predicted):<25} {expected:<14} {result}")

print(f"\n{correct}/{len(test_queries)} queries matched expected label.")


## What Happens After This Notebook

---

### 1. The trained model is in `models/intent_classifier/`

Contents:
- `pytorch_model.bin` — trained weights
- `config.json`, `tokenizer_config.json`, `vocab.txt` — model files
- `classifier_config.json` — labels, crisis threshold, evaluation results

---

### 2. Integrating with the chatbot

In `src/safety/escalation.py`, replace `_detect_type()` with a call to the
DistilBERT classifier:

```python
# Load once at module level (same caching pattern as embedder.py)
from transformers import AutoTokenizer, DistilBertForSequenceClassification
import torch, json

_config = json.load(open("models/intent_classifier/classifier_config.json"))
_clf_tokenizer = AutoTokenizer.from_pretrained("models/intent_classifier")
_clf_model = DistilBertForSequenceClassification.from_pretrained("models/intent_classifier")
_clf_model.eval()

def _detect_type(message: str) -> set:
    inputs = _clf_tokenizer(message, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        logits = _clf_model(**inputs).logits[0]
    probs = torch.sigmoid(logits).numpy()
    labels = ["educational", "diagnostic", "crisis"]
    result = set()
    for i, label in enumerate(labels):
        thresh = _config["crisis_threshold"] if label == "crisis" else 0.5
        if probs[i] >= thresh:
            result.add(label)
    return result
```

The **public interface** (`should_escalate`, `get_escalation_response`) stays
completely unchanged — only `_detect_type()` is swapped.

---

### 3. Retraining with updated data

1. Re-export from Label Studio
2. Run `python -m src.annotation.merge_annotations`
3. Re-run this notebook from **Cell 6** downward
4. No code changes needed

---

### 4. If crisis recall is below threshold

- **Lower `crisis_threshold`** using the recommended value from Cell 11
- Or **collect more crisis examples** from the Google Form and retrain
- Consider **class-weighted loss** if crisis remains severely underrepresented
